In [2]:
!pip install pinecone sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.1/225.1 kB 7.7 MB/s eta 0:00:00


In [4]:
from datasets import load_dataset

import pinecone
from pinecone import Pinecone, ServerlessSpec
from sentence_transformers import SentenceTransformer

In [5]:
fineweb = load_dataset("HuggingFaceFW/fineweb", name="sample-10BT", split="train", streaming=True)

README.md:   0%|          | 0.00/44.3k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/27468 [00:00<?, ?it/s]

In [5]:
fineweb

IterableDataset({
    features: ['text', 'id', 'dump', 'url', 'date', 'file_path', 'language', 'language_score', 'token_count'],
    num_shards: 15
})

In [6]:
PINECONE_API_KEY='pcsk_4XrPLQ_AmMoKRQXNpxFpLkRmCGWDetSDfZr8fnG3caeMhqLQFdsZcJWUc6XNREfsBznLGz'
pc = Pinecone(api_key=PINECONE_API_KEY)
indexes = pc.list_indexes()

model = SentenceTransformer('all-MiniLM-L6-v2')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [11]:
pc.create_index(
    name="text",
    dimension=model.get_sentence_embedding_dimension(),  # 384
    metric="cosine",
    spec=ServerlessSpec(cloud="aws", region="us-east-1")
)

/tmp/ipykernel_1070/453784310.py:3: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  dimension=model.get_sentence_embedding_dimension(),  # 384


Name:,text
Status:,Ready
Ready:,Yes
Deployment:,Managed (aws/us-east-1)
Host:,https://text-7mk7crn.svc.aped-4627-b74a.pinecone.io
Deletion Protection:,disabled
Schema fields:,2
Read capacity:,"ReadCapacityOnDemandResponse(status=ReadCapacityStatus(state='Ready', current_shards=None, current_replicas=None, error_message=None))"


In [7]:
index = pc.index(name="text")
index

Index(host='https://text-7mk7crn.svc.aped-4627-b74a.pinecone.io')

In [ ]:
# Let's embed our data and upsert to pinecone

# Define the number of items you want to process (subset size)
subset_size = 10000

vectors_to_upsert = []
for idx, item in enumerate(fineweb):
  if idx >= subset_size:
    break

  # Check https://huggingface.co/datasets/HuggingFaceFW/fineweb to see more keys
  text = item.get("text")
  id = str(item.get("id"))
  metadata = {"language": item.get("language")}

  # Create an embedding for the text
  embedding = model.encode(text, show_progress_bar=False).tolist()
  vectors_to_upsert.append((id, embedding, metadata))

In [11]:
# Upsert to Pinecone in batches
batch_size = 1000  # Adjust based on envirnonment and datasize
for i in range(0, len(vectors_to_upsert), batch_size):
    batch = vectors_to_upsert[i:i + batch_size]
    index.upsert(vectors=batch)
    print("Number:", i)

Number: 0
Number: 1000
Number: 2000
Number: 3000
Number: 4000
Number: 5000
Number: 6000
Number: 7000
Number: 8000
Number: 9000


In [13]:
index_stats = index.describe_index_stats()
print(index_stats)
# You can also specifically check the total vector count
print(f"Total vector count in index: {index_stats.total_vector_count}")

DescribeIndexStatsResponse(dimension=384, total_vector_count=10000, metric='cosine', namespaces=1)
Total vector count in index: 10000


In [14]:
len(vectors_to_upsert)

10000